# Exercice 4 - Expected Loss et CDO

125 noms, maturité 1 an, coupon 1, recovery = 0. Trois tranches. Modèle Vasicek.

In [ ]:
import numpy as np
from scipy.stats import norm, binom
from scipy.integrate import quad
import matplotlib.pyplot as plt

## Question 1 - Proba de défaut conditionnelle

Dans Vasicek, conditionnellement au facteur systémique F, les défauts sont indépendants. La proba de défaut conditionnelle à F=f c'est :

p(f) = Phi( (Phi^-1(PD) - sqrt(rho)*f) / sqrt(1-rho) )

Ca vient du fait que le defaut a lieu quand le rendement X_i = sqrt(rho)*F + sqrt(1-rho)*eps_i tombe en dessous du seuil Phi^-1(PD).

In [ ]:
def p_cond(f, PD, rho):
    return norm.cdf((norm.ppf(PD) - np.sqrt(rho)*f) / np.sqrt(1-rho))

## Question 2 - Loi binomiale

Conditionnellement à F, le nb de défauts suit une binomiale Bin(N, p(F)). Pour la proba non conditionnelle on intègre sur toutes les valeurs de F (qui est gaussien standard).

In [ ]:
N = 125
PD = 0.02
rho = 0.10

In [ ]:
def proba_k(k, N, PD, rho):
    """P(K=k) par integration sur le facteur"""
    def integ(f):
        pc = p_cond(f, PD, rho)
        return binom.pmf(k, N, pc) * norm.pdf(f)
    res, _ = quad(integ, -5, 5)
    return res

In [ ]:
# calcul de la distribution complete
dist = np.array([proba_k(k, N, PD, rho) for k in range(N+1)])
print(f"E[K] = {sum(k*dist[k] for k in range(N+1)):.2f} (attendu {N*PD:.1f})")

## Question 3 - EL par tranche

On découpe en tranches selon le nb de defauts :
- Equity : 0 à 4 defauts (0-3% du pool)
- Mezzanine : 4 à 9
- Senior : au dessus de 9

La perte de la tranche c'est : min(max(k - attachement, 0), taille de la tranche)

In [ ]:
# tranches
Ka_eq, Kd_eq = 0, 4
Ka_mz, Kd_mz = 4, 9
Ka_sr, Kd_sr = 9, 125

In [ ]:
def EL_tranche(Ka, Kd, distribution):
    taille = Kd - Ka
    el = 0
    for k in range(len(distribution)):
        loss = min(max(k - Ka, 0), taille)
        el += loss * distribution[k]
    return el

el_eq = EL_tranche(Ka_eq, Kd_eq, dist)
el_mz = EL_tranche(Ka_mz, Kd_mz, dist)
el_sr = EL_tranche(Ka_sr, Kd_sr, dist)

print(f"EL Equity     = {el_eq:.3f} soit {el_eq/(Kd_eq-Ka_eq)*100:.1f}% de la tranche")
print(f"EL Mezzanine  = {el_mz:.3f} soit {el_mz/(Kd_mz-Ka_mz)*100:.1f}% de la tranche")
print(f"EL Senior     = {el_sr:.4f} soit {el_sr/(Kd_sr-Ka_sr)*100:.3f}% de la tranche")
print(f"\nTotal = {el_eq+el_mz+el_sr:.3f} (devrait etre ≈ {N*PD:.1f})")

In [ ]:
# distribution graphique
plt.figure(figsize=(9, 4))
plt.bar(range(20), dist[:20], color='steelblue', alpha=0.7)
plt.axvline(Kd_eq, color='red', ls='--', label='Détach equity')
plt.axvline(Kd_mz, color='orange', ls='--', label='Détach mezz')
plt.xlabel('Nb de défauts')
plt.ylabel('Probabilité')
plt.title('Distribution des défauts')
plt.legend()
plt.show()

## Sensibilité à la corrélation

In [ ]:
print(f"{'rho':<6} {'EL Eq%':<10} {'EL Mz%':<10} {'EL Sr%':<10}")
print("-"*36)
for rho_test in [0.02, 0.05, 0.10, 0.20, 0.30]:
    d = np.array([proba_k(k, N, PD, rho_test) for k in range(N+1)])
    e1 = EL_tranche(Ka_eq, Kd_eq, d) / (Kd_eq - Ka_eq) * 100
    e2 = EL_tranche(Ka_mz, Kd_mz, d) / (Kd_mz - Ka_mz) * 100
    e3 = EL_tranche(Ka_sr, Kd_sr, d) / (Kd_sr - Ka_sr) * 100
    print(f"{rho_test:<6.2f} {e1:<10.1f} {e2:<10.2f} {e3:<10.4f}")

Quand rho augmente la distribution s'etale : la tranche equity perd moins en moyenne (plus de chance de zero defaut) mais les tranches hautes prennent plus cher (queue de distribution). C'est le principe du correlation trade.